In [2]:
import torch
from PIL import Image
from transformers import XLMRobertaTokenizer
from torchvision import transforms
from collections import Counter
from tqdm import tqdm
import pandas as pd
import os
import json
import torch.nn.functional as F

# --- 0. 실행 환경 설정 ---
# 사용하려는 GPU의 ID를 설정합니다. (예: "0" 또는 "0,1")
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

사용 장치: cuda


In [3]:
# --- 1. BEiT-3 모델 구조 import ---
# BEiT-3 레포지토리의 'modeling_finetune.py' 파일이 이 스크립트와 같은 디렉토리에 있어야 합니다.
try:
    from modeling_finetune import beit3_large_patch16_768_vqav2
except ImportError:
    print("오류: 'modeling_finetune.py'를 찾을 수 없습니다.")
    print("BEiT-3 레포지토리에서 해당 파일을 다운로드하여 이 스크립트와 같은 디렉토리에 위치시켜 주세요.")
    exit()

# --- 2. 경로 설정 (사용자 환경에 맞게 수정 필수) ---
# Fine-tuning된 모델 가중치 파일 경로
model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_indomain_patch16_768_vgqaaug_vqa.pth"
# 문장 토크나이저 모델 경로
spm_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
# VQAv2 답변 라벨 파일 경로
vqa_answer_label_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"
# 추론할 데이터 CSV 파일 경로
test_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/vqa/vqa_rephrased_by_llama_with_img_path.csv'
# 원본 이미지들이 있는 디렉토리 경로
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
# 최종 제출 파일 저장 경로
submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/beit_v3_vqa.csv'
# 원본 샘플 제출 파일 경로 (ID 순서 정렬용)
sample_submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv'

# --- 3. 모델 및 토크나이저 불러오기 ---
print("\n모델과 토크나이저를 로딩합니다...")
tokenizer = XLMRobertaTokenizer(spm_model_path)
model = beit3_large_patch16_768_vqav2()

if not os.path.exists(model_path):
    print(f"오류: 모델 가중치 파일을 찾을 수 없습니다 - {model_path}")
    exit()

checkpoint = torch.load(model_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()
print("모델 로딩 완료.")


모델과 토크나이저를 로딩합니다...
모델 로딩 완료.


In [4]:

# --- 4. VQAv2 답변 사전 생성 ('yes', 'no' 인덱스 확인) ---
print("\nVQAv2 답변 사전을 생성합니다...")
try:
    with open(vqa_answer_label_path, 'r') as f:
        vqa_data = json.load(f)
    raw_annotations = vqa_data['annotations']
    answer_counter = Counter(ann['multiple_choice_answer'] for ann in raw_annotations)
    num_top_answers = 3129
    top_answers = answer_counter.most_common(num_top_answers)
    answer_vocab = [answer for answer, count in top_answers]
    answer_to_idx = {answer: i for i, answer in enumerate(answer_vocab)}

    # 'yes'와 'no'의 인덱스를 미리 찾아둡니다. 이 로직의 핵심입니다.
    if 'yes' in answer_to_idx and 'no' in answer_to_idx:
        yes_idx = answer_to_idx['yes']
        no_idx = answer_to_idx['no']
        print(f"답변 사전 생성 완료. 'yes' 인덱스: {yes_idx}, 'no' 인덱스: {no_idx}")
    else:
        raise ValueError("'yes' 또는 'no'를 답변 사전에서 찾을 수 없습니다!")

except (FileNotFoundError, KeyError, ValueError) as e:
    print(f"답변 사전 생성 중 오류 발생: {e}")
    exit()

# --- 5. 데이터 로드 및 추론 준비 ---
print(f"\n추론할 데이터를 '{test_csv_path}'에서 로드합니다.")
if not os.path.exists(test_csv_path):
    print(f"오류: 테스트 CSV 파일을 찾을 수 없습니다 - {test_csv_path}")
    exit()
test_df = pd.read_csv(test_csv_path)

# 이미지 전처리(Transform) 정의
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

# --- 6. 추론 실행 (선택지별 'yes' 확률 비교) ---
print("\n추론을 시작합니다 (선택지별 'yes' 확률 비교 방식)...")
results = []
pbar = tqdm(test_df.iterrows(), total=len(test_df))

for _, row in pbar:
    image_path = os.path.join(image_dir, row['img_path'].lstrip('./'))

    if not os.path.exists(image_path):
        results.append({'ID': row['ID'], 'answer': 'A'}) # 이미지가 없으면 기본값 'A'
        continue
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        results.append({'ID': row['ID'], 'answer': 'A'}) # 오류 시 기본값 'A'
        continue

    choice_scores = {}
    # 각 선택지(A, B, C, D)에 대해 개별적으로 추론 실행
    for choice_key in ['A', 'B', 'C', 'D']:
        # 선택지 텍스트를 가져와 "Yes or no" 질문으로 구성
        choice_text = row[f'Rephrased_{choice_key}'] # 'Rephrased_A', 'Rephrased_B' 등의 열을 사용

        question_text = f"{choice_text}. Yes or no?"

        pbar.set_description(f"Processing ID: {row['ID']}, Choice: {choice_key}")
        
        # 텍스트 토크나이징
        encoding = tokenizer(
            question_text, padding='max_length', truncation=True,
            max_length=40, return_tensors='pt'
        )
        question_tokens = encoding['input_ids'].to(device)
        padding_mask = encoding['attention_mask'].to(device)
        
        # 모델 추론
        with torch.no_grad():
            output_logits = model(
                image=image_tensor,
                question=question_tokens,
                padding_mask=padding_mask
            )

        # 'yes'와 'no'의 점수(logit) 추출
        yes_logit = output_logits[0, yes_idx].item()
        no_logit = output_logits[0, no_idx].item()

        # 소프트맥스를 적용하여 'yes'일 확률 계산
        probabilities = F.softmax(torch.tensor([no_logit, yes_logit]), dim=0)
        yes_prob = probabilities[1].item()
        
        # 해당 선택지의 점수를 'yes' 확률로 저장
        choice_scores[choice_key] = yes_prob

    # 가장 높은 'yes' 확률을 가진 선택지를 최종 정답으로 선택
    best_choice = max(choice_scores, key=choice_scores.get)
    results.append({'ID': row['ID'], 'answer': best_choice})


VQAv2 답변 사전을 생성합니다...
답변 사전 생성 완료. 'yes' 인덱스: 0, 'no' 인덱스: 1

추론할 데이터를 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/vqa/vqa_rephrased_by_llama_with_img_path.csv'에서 로드합니다.

추론을 시작합니다 (선택지별 'yes' 확률 비교 방식)...


Processing ID: TEST_059, Choice: D: 100%|██████████| 60/60 [00:38<00:00,  1.57it/s]


In [5]:
# --- 7. 최종 결과 제출 파일로 저장 ---
print("\n추론 완료! 최종 제출 파일을 생성합니다.")
final_submission_df = pd.DataFrame(results)

# 원본 제출 파일 형식에 맞게 ID 순으로 정렬 (권장)
try:
    sample_df = pd.read_csv(sample_submission_path)
    final_submission_df = final_submission_df.set_index('ID').loc[sample_df['ID']].reset_index()
    print("\n샘플 제출 파일에 맞게 ID 순서 정렬 완료.")
except Exception as e:
    print(f"\nWarning: ID 순서 정렬 중 오류 발생: {e}. 정렬을 건너뜁니다.")

# 최종 결과 저장
output_dir = os.path.dirname(submission_path)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

final_submission_df.to_csv(submission_path, index=False)

print(f"\n✅ 모든 작업 완료! 최종 제출 파일이 '{submission_path}'에 저장되었습니다.")
print("\n--- 최종 제출 파일 미리보기 (상위 5개) ---")
print(final_submission_df.head())


추론 완료! 최종 제출 파일을 생성합니다.

샘플 제출 파일에 맞게 ID 순서 정렬 완료.

✅ 모든 작업 완료! 최종 제출 파일이 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/beit_v3_vqa.csv'에 저장되었습니다.

--- 최종 제출 파일 미리보기 (상위 5개) ---
         ID answer
0  TEST_000      C
1  TEST_001      A
2  TEST_002      A
3  TEST_003      A
4  TEST_004      B
